# Generación de audio en GPU (Colab) — Banco MusicCaps

Genera audio para los 100 prompts de `testset_metadata.csv` con uno o varios de estos modelos:

| Modelo | HF ID | Notas |
|---|---|---|
| **MusicGen-small** | `facebook/musicgen-small` | ~300 MB · abierto |
| **MusicGen-medium** | `facebook/musicgen-medium` | ~1.5 GB · abierto |
| **AudioLDM2** | `cvssp/audioldm2` | ~3 GB · abierto · difusión |
| **Stable Audio Open** | `stabilityai/stable-audio-open-1.0` | ~3 GB · **requiere token HF + aceptar licencia** |

**Antes de empezar:** menú `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.

Salida: un archivo `wavs.zip` con `wavs/<modelo>/<id>.wav` (32 kHz mono, 10 s), listo para descargar y evaluar (FAD/CLAP/KLD/KAD) en tu máquina.

## 1 · Verificar GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No hay GPU. Ve a Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU y reinicia.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2 · Instalar dependencias
Tarda ~2-3 min. Si Colab pide reiniciar el entorno, hazlo y vuelve a correr desde aquí.

In [ ]:
%pip install -q -U "transformers>=4.40" "diffusers>=0.27" accelerate torchaudio soundfile sentencepiece

## 3 · Subir `testset_metadata.csv`
Selecciona el archivo desde tu máquina (`/Users/homer/Downloads/hybrid_engine/testset_metadata.csv`).

In [ ]:
import csv
from google.colab import files

uploaded = files.upload()  # elige testset_metadata.csv
csv_name = next(iter(uploaded))

with open(csv_name, encoding='utf-8') as f:
    metadata = list(csv.DictReader(f))
print(f'{len(metadata)} prompts cargados. Ejemplo:')
print(' ', metadata[0]['id'], '->', metadata[0]['caption'][:90], '...')

## 4 · Configuración
Elige qué modelos generar. Empieza con `musicgen-small` para validar el flujo (rápido), luego añade el resto.

In [ ]:
#@title Modelos a generar { run: "auto" }
musicgen_small  = True   #@param {type:"boolean"}
musicgen_medium = False  #@param {type:"boolean"}
audioldm2       = False  #@param {type:"boolean"}
stable_audio    = False  #@param {type:"boolean"}

MODELS = []
if musicgen_small:  MODELS.append('musicgen-small')
if musicgen_medium: MODELS.append('musicgen-medium')
if audioldm2:       MODELS.append('audioldm2')
if stable_audio:    MODELS.append('stable-audio-open')
print('Modelos seleccionados:', MODELS)

# Constantes de generación (mismas que el repo)
TARGET_DURATION = 10.0
TARGET_SR = 32000
TARGET_DBFS = -3.0
SEED = 42

### (Opcional) Token HF — solo si usas Stable Audio Open
Es un modelo *gated*: acepta la licencia en https://huggingface.co/stabilityai/stable-audio-open-1.0 y crea un token en https://huggingface.co/settings/tokens . Salta esta celda si no lo usas.

In [ ]:
from huggingface_hub import login
login()  # pega tu token cuando lo pida

## 5 · Funciones de generación y post-proceso

In [ ]:
import gc
from pathlib import Path
import torch, torchaudio

DEVICE = 'cuda'
WAVS = Path('wavs'); WAVS.mkdir(exist_ok=True)

def peak_normalize(wav, dbfs=TARGET_DBFS):
    peak = wav.abs().max()
    if peak > 0:
        return (wav / peak) * (10 ** (dbfs / 20.0))
    return wav

def resample(wav, orig_sr, target_sr=TARGET_SR):
    if orig_sr != target_sr:
        return torchaudio.transforms.Resample(orig_sr, target_sr)(wav)
    return wav

def pad_or_trim(wav, n=int(TARGET_DURATION * TARGET_SR)):
    """Garantiza exactamente n muestras (10 s a 32 kHz = 320000)."""
    if wav.shape[-1] < n:
        wav = torch.nn.functional.pad(wav, (0, n - wav.shape[-1]))
    return wav[..., :n]

def clear():
    gc.collect(); torch.cuda.empty_cache()

def save(wav, path):
    wav = pad_or_trim(peak_normalize(wav))
    torchaudio.save(str(path), wav, TARGET_SR)


def gen_musicgen(model_id, out_dir):
    from transformers import AutoProcessor, MusicgenForConditionalGeneration
    print(f'Cargando {model_id} ...')
    proc = AutoProcessor.from_pretrained(model_id)
    model = MusicgenForConditionalGeneration.from_pretrained(model_id).to(DEVICE)
    max_new = int(256 * (TARGET_DURATION / 5.0))
    for row in metadata:
        out = out_dir / f"{row['id']}.wav"
        if out.exists():
            continue
        inputs = proc(text=[row['caption']], padding=True, return_tensors='pt').to(DEVICE)
        torch.manual_seed(SEED)
        vals = model.generate(**inputs, do_sample=True, guidance_scale=3.0,
                              max_new_tokens=max_new, temperature=1.0, top_k=250)
        wav = vals[0, 0].cpu()
        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
        # MusicGen sale a 32 kHz
        save(resample(wav, model.config.audio_encoder.sampling_rate), out)
        print(f"  {out.name}")
    del model, proc; clear()


def gen_audioldm2(model_id, out_dir):
    from diffusers import AudioLDM2Pipeline
    print(f'Cargando {model_id} ...')
    pipe = AudioLDM2Pipeline.from_pretrained(model_id, torch_dtype=torch.float16).to(DEVICE)
    for row in metadata:
        out = out_dir / f"{row['id']}.wav"
        if out.exists():
            continue
        g = torch.Generator(DEVICE).manual_seed(SEED)
        audio = pipe(row['caption'], num_inference_steps=200,
                     audio_length_in_s=TARGET_DURATION, guidance_scale=3.5,
                     generator=g).audios[0]
        wav = torch.tensor(audio).unsqueeze(0)  # AudioLDM2 sale a 16 kHz
        save(resample(wav, 16000), out)
        print(f"  {out.name}")
    del pipe; clear()


def gen_stable_audio(model_id, out_dir):
    from diffusers import StableAudioPipeline
    print(f'Cargando {model_id} ...')
    pipe = StableAudioPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to(DEVICE)
    for row in metadata:
        out = out_dir / f"{row['id']}.wav"
        if out.exists():
            continue
        g = torch.Generator(DEVICE).manual_seed(SEED)
        res = pipe(row['caption'], negative_prompt='Low quality, noise, distortion.',
                   num_inference_steps=200, audio_end_in_s=TARGET_DURATION,
                   num_waveforms_per_prompt=1, generator=g)
        audio = res.audios[0]  # [canales, muestras] a 44100 Hz
        wav = torch.from_numpy(audio).mean(dim=0, keepdim=True)  # mono
        save(resample(wav, 44100), out)
        print(f"  {out.name}")
    del pipe; clear()


GENERATORS = {
    'musicgen-small':    lambda d: gen_musicgen('facebook/musicgen-small', d),
    'musicgen-medium':   lambda d: gen_musicgen('facebook/musicgen-medium', d),
    'audioldm2':         lambda d: gen_audioldm2('cvssp/audioldm2', d),
    'stable-audio-open': lambda d: gen_stable_audio('stabilityai/stable-audio-open-1.0', d),
}
print('Funciones listas.')

## 6 · Generar
Tiempos aproximados en T4 para los 100 clips: MusicGen-small ~8 min, medium ~20 min, AudioLDM2/Stable Audio ~30-40 min cada uno. Es seguro reanudar: los `.wav` ya generados se omiten.

In [ ]:
import time
for name in MODELS:
    out_dir = WAVS / name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n=== {name} ===')
    t0 = time.time()
    GENERATORS[name](out_dir)
    print(f'{name} listo en {(time.time()-t0)/60:.1f} min  ->  {len(list(out_dir.glob("*.wav")))} wav')

## 7 · Verificar y empaquetar

In [ ]:
import soundfile as sf
ok = bad = 0
for wav in sorted(WAVS.rglob('*.wav')):
    info = sf.info(str(wav))
    good = info.frames == 320000 and info.samplerate == 32000 and info.channels == 1
    ok += good; bad += (not good)
    if not good:
        print('  PROBLEMA:', wav, info.frames, info.samplerate, info.channels)
print(f'{ok} correctos · {bad} con problemas')

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('wavs', 'zip', '.', 'wavs')
print('Descargando wavs.zip ...')
files.download('wavs.zip')

## 8 · Evaluación de métricas (FAD / CLAP / KLD / KAD)

Calcula las **cuatro métricas reales** comparando cada modelo contra el banco de referencia `testset/`.
Las fórmulas son idénticas a las del repo:

- **FAD** — distancia de Fréchet sobre embeddings VGGish (menor = mejor).
- **CLAP** — similitud coseno audio↔prompt con laion-clap (mayor = mejor).
- **KLD** — divergencia KL sobre probabilidades PaSST/AudioSet (menor = mejor).
- **KAD** — MMD (kernel RBF, ancho por mediana) sobre los mismos embeddings VGGish (menor = mejor).

**Necesitas el banco de referencia.** En tu Mac:
```bash
zip -r testset.zip testset
```
y sube `testset.zip` en la siguiente celda.

In [ ]:
# Subir y descomprimir testset.zip (banco de referencia real)
import io, zipfile
from pathlib import Path
from google.colab import files

up = files.upload()  # elige testset.zip
name = next(iter(up))
with zipfile.ZipFile(io.BytesIO(up[name])) as z:
    z.extractall('.')
n = len(list(Path('testset').glob('*.wav')))
print('testset/ ->', n, 'wav de referencia')

### Instalar dependencias de métricas
Corre esto **después** de generar (laion-clap puede fijar una versión distinta de `transformers`).
VGGish, CLAP y PaSST descargan sus checkpoints automáticamente la primera vez.

In [ ]:
# FAD/KAD (VGGish) + CLAP (laion-clap)
%pip install -q torchvggish laion-clap

In [ ]:
# KLD (PaSST). Opcional: si falla, las otras 3 métricas igual se calculan.
%pip install -q hear21passt

### Funciones de métricas (portadas 1:1 del repo)

In [ ]:
import numpy as np
from scipy import linalg
from scipy.spatial.distance import pdist
from pathlib import Path

AUDIO_EXTS = ('.wav', '.flac', '.mp3', '.ogg')
def list_audio(folder):
    folder = Path(folder)
    return sorted(p for p in folder.rglob('*') if p.suffix.lower() in AUDIO_EXTS)

# --- FAD (fad_score.py) ---
def frechet_distance(mu_r, sigma_r, mu_g, sigma_g, eps=1e-6):
    diff = mu_r - mu_g
    term1 = float(diff @ diff)
    product = sigma_r @ sigma_g
    product = product + np.eye(product.shape[0]) * eps
    res = linalg.sqrtm(product)
    sqrt_prod = res[0] if isinstance(res, tuple) else res  # scipy <1.18 vs >=1.18
    if np.iscomplexobj(sqrt_prod):
        sqrt_prod = sqrt_prod.real
    term2 = float(np.trace(sigma_r + sigma_g - 2.0 * sqrt_prod))
    return term1 + term2

def stats(emb):
    return np.mean(emb, axis=0), np.cov(emb, rowvar=False)

# --- KAD / MMD (kad_score.py) ---
def median_bandwidth(ref):
    ref = np.asarray(ref, np.float64)
    if ref.ndim != 2 or ref.shape[0] < 2:
        return 1.0
    d = pdist(ref, 'euclidean')
    if d.size == 0:
        return 1.0
    s = float(np.median(d))
    return s if s > 1e-12 else 1.0

def rbf_gram(a, b, gamma):
    a_sq = np.sum(a * a, axis=1)[:, None]
    b_sq = np.sum(b * b, axis=1)[None, :]
    sq = np.maximum(a_sq + b_sq - 2.0 * (a @ b.T), 0.0)
    return np.exp(-gamma * sq)

def mmd2_unbiased(real, gen, sigma):
    real = np.asarray(real, np.float64)
    gen = np.asarray(gen, np.float64)
    n, m = real.shape[0], gen.shape[0]
    gamma = 1.0 / (2.0 * sigma * sigma)
    kxx = rbf_gram(real, real, gamma)
    kyy = rbf_gram(gen, gen, gamma)
    kxy = rbf_gram(real, gen, gamma)
    sxx = (kxx.sum() - np.trace(kxx)) / (n * (n - 1))
    syy = (kyy.sum() - np.trace(kyy)) / (m * (m - 1))
    sxy = kxy.sum() / (n * m)
    return float(sxx + syy - 2.0 * sxy)

# --- KLD (kld_metric.py) ---
def kld_from_dists(p, q, eps=1e-12):
    p = np.clip(np.asarray(p, np.float64), eps, None); p = p / p.sum()
    q = np.clip(np.asarray(q, np.float64), eps, None); q = q / q.sum()
    return float(np.sum(p * np.log(p / q)))

print('Metricas (math) listas.')

In [ ]:
import torch
DEVICE = 'cuda'

# VGGish -> FAD (todos los frames apilados) y KAD (un vector por clip)
_vggish = None
def vggish_frames(path):
    global _vggish
    if _vggish is None:
        import torchvggish
        m = torchvggish.vggish(); m.eval(); m.to(DEVICE)
        _vggish = (m, torchvggish.vggish_input)
    m, vin = _vggish
    ex = vin.wavfile_to_examples(str(path))
    t = torch.tensor(ex, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        emb = m(t)
    emb = emb.detach().cpu().numpy() if hasattr(emb, 'detach') else np.asarray(emb)
    return emb if emb.ndim == 2 else emb[None, :]

def vggish_collect(files):
    frames, pooled = [], []
    for f in files:
        e = vggish_frames(f)
        frames.append(e)
        pooled.append(e.mean(axis=0))
    return np.vstack(frames), np.vstack(pooled)

# CLAP (laion-clap)
_clap = None
def clap():
    global _clap
    if _clap is None:
        import laion_clap
        _clap = laion_clap.CLAP_Module(enable_fusion=False, device=DEVICE)
        _clap.load_ckpt()
    return _clap

def clap_audio_emb(path):
    return clap().get_audio_embedding_from_filelist(x=[str(path)], use_tensor=False)

def clap_text_emb(text):
    return clap().get_text_embedding([text], use_tensor=False)

def cosine(a, b):
    a = a.ravel(); b = b.ravel()
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    return 0.0 if na == 0 or nb == 0 else float(a @ b / (na * nb))

# PaSST (hear21passt) -> KLD
import librosa
_passt = None
def passt_proba(path):
    global _passt
    if _passt is None:
        from hear21passt.base import get_basic_model
        _passt = get_basic_model(mode='logits').to(DEVICE); _passt.eval()
    y, _sr = librosa.load(str(path), sr=32000, mono=True)
    t = torch.tensor(y, dtype=torch.float32, device=DEVICE)[None, :]
    with torch.no_grad():
        logits = _passt(t)
    logits = logits.squeeze(0).detach().cpu().numpy().astype(np.float64)
    logits = logits - logits.max()
    e = np.exp(logits); s = e.sum()
    return e / s if s > 0 else np.full_like(e, 1.0 / len(e))

print('Extractores listos (VGGish / CLAP / PaSST).')

### Calcular y mostrar la tabla comparativa real
Tarda ~5-10 min por modelo (VGGish + CLAP + PaSST sobre 100+100 clips).

In [ ]:
import pandas as pd

REAL_DIR = Path('testset')
WAVS = Path('wavs')
real_files = list_audio(REAL_DIR)
assert real_files, 'Falta testset/. Sube y descomprime testset.zip arriba.'
id2cap   = {r['id']: r['caption'] for r in metadata}
id2genre = {r['id']: r['genre'] for r in metadata}
id2tempo = {r['id']: r.get('tempo_bpm', '') for r in metadata}

RUN_KLD = True  # ponlo en False si PaSST no se instalo

print('VGGish sobre referencia (FAD/KAD)...')
real_frames, real_pooled = vggish_collect(real_files)
mu_r, sig_r = stats(real_frames)
sigma_bw = median_bandwidth(real_pooled)

p_real = None
if RUN_KLD:
    try:
        print('PaSST sobre referencia (KLD)...')
        p_real = np.mean([passt_proba(f) for f in real_files], axis=0)
    except Exception as exc:
        print('KLD desactivado (PaSST no disponible):', exc); RUN_KLD = False

set_rows, clip_rows = [], []
for model in MODELS:
    gen_files = list_audio(WAVS / model)
    if not gen_files:
        print('Sin wavs para', model, '- omitido'); continue
    print('===', model, '(', len(gen_files), 'clips )')
    gen_frames, gen_pooled = vggish_collect(gen_files)
    mu_g, sig_g = stats(gen_frames)
    fad = frechet_distance(mu_r, sig_r, mu_g, sig_g)
    kad = mmd2_unbiased(real_pooled, gen_pooled, sigma_bw)

    # PaSST por clip una sola vez (sirve para KLD de conjunto y por clip)
    gen_probs = {}
    if RUN_KLD:
        for f in gen_files:
            gen_probs[f.stem] = passt_proba(f)
        p_gen = np.mean(list(gen_probs.values()), axis=0)
        kld_set = kld_from_dists(p_real, p_gen)
    else:
        kld_set = float('nan')

    sims = []
    for f in gen_files:
        cap = id2cap.get(f.stem)
        if cap is None:
            continue
        clap = cosine(clap_audio_emb(f), clap_text_emb(cap))
        sims.append(clap)
        # passt_kld por clip = divergencia de ESTE clip vs la referencia media
        # (proxy por-clip; la KLD de conjunto va en set_level)
        kld_clip = kld_from_dists(gen_probs[f.stem], p_real) if RUN_KLD else ''
        clip_rows.append({
            'model': model,
            'clip_id': model + '_' + f.stem,
            'genre': id2genre.get(f.stem, 'unknown'),
            'tempo_bpm': id2tempo.get(f.stem, ''),
            'clap_score': round(clap, 6),
            'passt_kld': (round(kld_clip, 6) if kld_clip != '' else ''),
        })
    sims = np.array(sims)
    set_rows.append({'model': model,
                     'fad_vggish': round(fad, 4),
                     'mean_clap': round(float(sims.mean()), 4),
                     'std_clap': round(float(sims.std()), 4),
                     'kld': (round(kld_set, 4) if kld_set == kld_set else ''),
                     'kad': round(kad, 6)})
    print('  FAD=%.4f  CLAP=%.4f+/-%.4f  KLD=%s  KAD=%.6f' % (fad, sims.mean(), sims.std(), kld_set, kad))

set_df = pd.DataFrame(set_rows)
clip_df = pd.DataFrame(clip_rows)
set_df.to_csv('real_set_level_metrics.csv', index=False)
clip_df.to_csv('real_clip_level_metrics.csv', index=False)
print('Guardados: real_set_level_metrics.csv (', len(set_df), 'modelos ) y real_clip_level_metrics.csv (', len(clip_df), 'clips )')
set_df

In [ ]:
from google.colab import files
files.download('real_set_level_metrics.csv')
files.download('real_clip_level_metrics.csv')

## 9 · Integrar los numeros reales en el repo
El notebook produce **dos CSV reales**:

- `real_set_level_metrics.csv` -> columnas `model, fad_vggish, mean_clap, std_clap, kld, kad`
- `real_clip_level_metrics.csv` -> columnas `model, clip_id, genre, tempo_bpm, clap_score, passt_kld`

En tu Mac, sustituye los datos sinteticos por los reales:

1. Copia los CSV al repo:
   ```bash
   cp real_set_level_metrics.csv  results/set_level_metrics.csv
   cp real_clip_level_metrics.csv results/clip_level_metrics.csv
   ```
2. Re-genera los 3 productos del benchmark (tabla + Kruskal-Wallis/Dunn por genero + radar):
   ```bash
   cd /Users/homer/Downloads/hybrid_engine
   source .venv-mac/bin/activate
   python scripts/analyze_results.py --results results/
   ```
3. Actualiza la tabla de `results/benchmark_section.tex` con estos valores
   (hasta ahora venia de `make_synthetic_results.py`).

**Notas:**
- El analisis por genero (Kruskal-Wallis + Dunn) usa `clap_score` por clip, que ya es real.
- `passt_kld` por clip es un *proxy* (divergencia de cada clip respecto a la referencia media);
  la KLD de conjunto, que es la metrica del paper, esta en `set_level_metrics.csv`.

## 10 · Modo servidor: conectar con tu frontend
En lugar de descargar el ZIP a mano, esta seccion levanta un **microservicio** en Colab
(FastAPI) detras de un **tunel publico** (`cloudflared`, sin registro). Tu frontend local
apunta su "Backend en la nube (GPU)" a esa URL y el boton **Generar** del Paso 4 corre en
la GPU de Colab, con progreso en vivo y descarga directa del `.zip`.

Requisitos: haber corrido las celdas **3** (subir CSV) y **5** (funciones de generacion).

In [ ]:
%pip install -q fastapi "uvicorn[standard]" nest_asyncio

### Definir el microservicio (mismo contrato de jobs que el backend local)

### Subir `benchmark_analysis.py` (para la tabla/radar/Kruskal en la nube)
Esta en tu repo: `hybrid_music_engine/new_metrics/benchmark_analysis.py`. Solo necesita numpy+scipy.

In [ ]:
from google.colab import files
print('Sube hybrid_music_engine/new_metrics/benchmark_analysis.py')
files.upload()  # queda en el cwd, importable como: from benchmark_analysis import benchmark_payload

In [ ]:
import threading, uuid, shutil
from pathlib import Path
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel

api = FastAPI()
api.add_middleware(CORSMiddleware, allow_origins=['*'],
                   allow_methods=['*'], allow_headers=['*'])
JOBS = {}
VALID = {'musicgen-small', 'musicgen-medium', 'audioldm2', 'stable-audio-open'}

class GenReq(BaseModel):
    model_name: str

def _run_job(job_id, model):
    j = JOBS[job_id]
    try:
        j.update(status='running', stage='generando',
                 message='Generando con ' + model + ' (' + str(len(metadata)) + ' clips)...',
                 progress=0.1)
        out = WAVS / model
        out.mkdir(parents=True, exist_ok=True)
        GENERATORS[model](out)            # reutiliza la generacion de la celda 5
        shutil.make_archive(model, 'zip', WAVS, model)
        n = len(list(out.glob('*.wav')))
        JOBS[job_id]['_zip'] = model + '.zip'
        j.update(status='completed', stage='listo',
                 message=str(n) + ' clips generados.', progress=1.0,
                 payload={'model': model, 'clips': n,
                          'download_url': '/download/' + job_id})
    except Exception as exc:
        j.update(status='failed', stage='error', message=str(exc), progress=0.0)

@api.get('/api/health')
def health():
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    return {'ok': True, 'gpu': name, 'models': sorted(VALID)}

@api.post('/api/jobs/generate-pretrained')
def gen(req: GenReq):
    if req.model_name not in VALID:
        return JSONResponse(status_code=422, content={'detail': 'modelo invalido'})
    jid = uuid.uuid4().hex[:12]
    JOBS[jid] = {'job_id': jid, 'status': 'queued', 'stage': 'en cola',
                 'message': 'En cola', 'progress': 0.0, 'payload': {}}
    threading.Thread(target=_run_job, args=(jid, req.model_name), daemon=True).start()
    return {'job_id': jid, 'mode': 'cloud-thread'}

@api.get('/api/jobs/{job_id}')
def job(job_id: str):
    return JOBS.get(job_id, {'status': 'failed', 'stage': 'error',
                             'message': 'job no encontrado', 'progress': 0.0})

@api.get('/download/{job_id}')
def download(job_id: str):
    z = JOBS.get(job_id, {}).get('_zip')
    if not z:
        return JSONResponse(status_code=404, content={'detail': 'sin archivo'})
    return FileResponse(z, filename=Path(z).name)

# --- Metricas en la nube (FAD/CLAP/KLD/KAD) ---
import pandas as pd
LAST_BENCHMARK = {}

def compute_benchmark_csvs():
    out = Path('results'); out.mkdir(exist_ok=True)
    real_files = list_audio(REAL_DIR)
    real_frames, real_pooled = vggish_collect(real_files)
    mu_r, sig_r = stats(real_frames); sigma_bw = median_bandwidth(real_pooled)
    p_real = np.mean([passt_proba(f) for f in real_files], axis=0)
    id2cap = {r['id']: r['caption'] for r in metadata}
    id2genre = {r['id']: r['genre'] for r in metadata}
    id2tempo = {r['id']: r.get('tempo_bpm', '') for r in metadata}
    set_rows, clip_rows = [], []
    for model in MODELS:
        gen_files = list_audio(WAVS / model)
        if not gen_files:
            continue
        gf, gp = vggish_collect(gen_files)
        fad = frechet_distance(mu_r, sig_r, *stats(gf))
        kad = mmd2_unbiased(real_pooled, gp, sigma_bw)
        gen_probs = {f.stem: passt_proba(f) for f in gen_files}
        p_gen = np.mean(list(gen_probs.values()), axis=0)
        kld_set = kld_from_dists(p_real, p_gen)
        sims = []
        for f in gen_files:
            cap = id2cap.get(f.stem)
            if cap is None:
                continue
            c = cosine(clap_audio_emb(f), clap_text_emb(cap)); sims.append(c)
            clip_rows.append({'model': model, 'clip_id': model + '_' + f.stem,
                              'genre': id2genre.get(f.stem, 'unknown'),
                              'tempo_bpm': id2tempo.get(f.stem, ''),
                              'clap_score': round(c, 6),
                              'passt_kld': round(kld_from_dists(gen_probs[f.stem], p_real), 6)})
        sims = np.array(sims)
        set_rows.append({'model': model, 'fad_vggish': round(fad, 4),
                         'mean_clap': round(float(sims.mean()), 4),
                         'std_clap': round(float(sims.std()), 4),
                         'kld': round(kld_set, 4), 'kad': round(kad, 6)})
    pd.DataFrame(set_rows).to_csv(out / 'set_level_metrics.csv', index=False)
    pd.DataFrame(clip_rows).to_csv(out / 'clip_level_metrics.csv', index=False)
    return out

def _run_metrics_job(job_id):
    j = JOBS[job_id]
    try:
        j.update(status='running', stage='metricas',
                 message='Calculando FAD/CLAP/KLD/KAD en GPU...', progress=0.1)
        out = compute_benchmark_csvs()
        from benchmark_analysis import benchmark_payload   # modulo subido (celda anterior)
        payload = benchmark_payload(out / 'set_level_metrics.csv', out / 'clip_level_metrics.csv')
        global LAST_BENCHMARK
        LAST_BENCHMARK = payload
        n = len(payload.get('table', {}).get('rows', []))
        j.update(status='completed', stage='listo',
                 message=str(n) + ' modelos evaluados.', progress=1.0, payload={'models': n})
    except Exception as exc:
        j.update(status='failed', stage='error', message=str(exc), progress=0.0)

@api.post('/api/jobs/eval-metrics')
def eval_metrics():
    jid = uuid.uuid4().hex[:12]
    JOBS[jid] = {'job_id': jid, 'status': 'queued', 'stage': 'en cola',
                 'message': 'En cola', 'progress': 0.0, 'payload': {}}
    threading.Thread(target=_run_metrics_job, args=(jid,), daemon=True).start()
    return {'job_id': jid, 'mode': 'cloud-thread'}

@api.get('/api/metrics/benchmark')
def benchmark():
    if LAST_BENCHMARK:
        return LAST_BENCHMARK
    s = Path('results/set_level_metrics.csv'); c = Path('results/clip_level_metrics.csv')
    if s.exists() and c.exists():
        from benchmark_analysis import benchmark_payload
        return benchmark_payload(s, c)
    return JSONResponse(status_code=404,
                        content={'detail': 'aun no se han calculado metricas en la nube'})

print('Microservicio definido (generacion + metricas).')

### Arrancar uvicorn (en segundo plano)

In [ ]:
import nest_asyncio, threading, time, uvicorn
nest_asyncio.apply()

def _serve():
    uvicorn.run(api, host='0.0.0.0', port=8100, log_level='warning')

threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)
print('uvicorn escuchando en el puerto 8100')

### Abrir el tunel publico y obtener la URL
Copia la URL `https://....trycloudflare.com` y pegala en el frontend:
**Paso 4 -> selecciona un modelo HF -> campo Backend en la nube (GPU) -> Conectar.**

In [ ]:
import subprocess, re, time, os

if not os.path.exists('cloudflared'):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

subprocess.Popen('./cloudflared tunnel --url http://localhost:8100 --no-autoupdate > cf.log 2>&1',
                 shell=True)

url = None
for _ in range(40):
    time.sleep(2)
    try:
        txt = open('cf.log').read()
    except FileNotFoundError:
        continue
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        break

print('=' * 50)
print('URL publica del backend GPU:', url)
print('Opcion A (local): Paso 4 -> Backend en la nube -> pega la URL -> Conectar.')
if url:
    print('Opcion B (sitio desplegado): abre tu sitio con este enlace 1-click:')
    print('   https://TU-SITIO.netlify.app/?cloud=' + url)
print('Manten el notebook abierto mientras generas.')
print('=' * 50)

> **Notas**
> - El tunel `trycloudflare` es temporal: si reinicias, genera una URL nueva (vuelve a pegarla).
> - El frontend en `http://localhost:5173` puede llamar a la URL `https://...` sin problema (CORS abierto; https desde http esta permitido).
> - El `.zip` se descarga directo desde el boton que aparece en el Paso 4 al terminar.
> - Esto cubre la **generacion**; las metricas se calculan en la seccion 8.